# M6-T3 — Train & Serialize the URL Char-CNN (Primary URL Model)

**Owner:** Sanjeewa Narayana  
**Depends on:** M6-T1 — frozen splits must be present and checksummed before running.

### Where to run
| Option | Setup | Training time |
|---|---|---|
| **Google Colab (recommended)** | `Runtime → Change runtime type → GPU (T4)` | ~11 min |
| **Local Jupyter with GPU** | `python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"` | ~11 min |
| **Local Jupyter CPU-only** | No setup needed — training still completes | ~2 hr |

### Inputs / Outputs
| | |
|---|---|
| **In** | `url_train.csv` / `url_test.csv` — raw `url` column only (no engineered features) |
| **Architecture** | Embedding(VOCAB,32) → Conv1D(128, k=3/4/5) → GlobalMaxPool → Concat → Dropout(0.4) → Dense(64) → Dense(1,sigmoid) |
| **Out** | `models/url_charcnn.keras` — trained model |
| | `models/url_char_vocab.json` — `char2idx` map + MAXLEN + VOCAB (mandatory for inference) |
| **S3** | `s3://email-security-pipeline-datasets/models/artifacts/url/url_charcnn.keras` |
| | `s3://email-security-pipeline-datasets/models/artifacts/url/url_char_vocab.json` |

### Acceptance criteria
- Test F1 ≥ 0.974 (reproduces M5-T8 result)
- Model artifact + `char_vocab.json` saved and uploaded to S3
- SHA-256 of both artifacts recorded

## Step 0 — Get the data

**Option A — Local Jupyter** (skip if files already in `data/processed/`):
```bash
mkdir -p data/processed
for f in url_train url_test; do
  aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/${f}.csv \
            data/processed/${f}.csv --profile lab-user
done
# verify checksums
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/SPLITS.sha256 \
          data/processed/SPLITS.sha256 --profile lab-user
cd data/processed && sha256sum -c SPLITS.sha256 --ignore-missing
```

**Option B — Google Colab** (run as `!` cells after connecting to GPU runtime):
```python
# Install AWS CLI if not present
!pip install -q awscli

# Configure credentials (run once per session)
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Download splits
!aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/url_train.csv url_train.csv
!aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/url_test.csv  url_test.csv
```
> ⚠️ Never paste credentials directly into the notebook. Use Colab Secrets (`🔑` sidebar) or enter them at runtime via `input()`.

In [2]:
import hashlib
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

MAXLEN       = 200
RANDOM_STATE = 42
S3_BASE      = 's3://email-security-pipeline-datasets/models/artifacts/url'
S3_PROFILE   = 'lab-user'

# Works both locally (data/processed/) and in Colab (cwd)
PROC      = Path('data/processed') if Path('data/processed/url_train.csv').exists() else Path('.')
MODELS    = Path('models')
MODELS.mkdir(exist_ok=True)

for f in ('url_train.csv', 'url_test.csv'):
    assert (PROC / f).exists(), f'Missing {f} — run Step 0 first'
print('Split files present.')

Split files present.


## Step 1 — Load frozen splits

In [2]:
tr = pd.read_csv(PROC / 'url_train.csv')
te = pd.read_csv(PROC / 'url_test.csv')
tr['url'] = tr['url'].astype(str)
te['url'] = te['url'].astype(str)

print(f'Train : {len(tr):>7,} rows  |  label dist: {tr["label"].value_counts().to_dict()}')
print(f'Test  : {len(te):>7,} rows  |  label dist: {te["label"].value_counts().to_dict()}')

Train : 512,895 rows  |  label dist: {0: 342464, 1: 170431}
Test  : 128,224 rows  |  label dist: {0: 85616, 1: 42608}


## Step 2 — Build character vocabulary & encode
Vocab is built from **training URLs only** to avoid test leakage. Index 0 is reserved for padding / unknown characters.

In [3]:
np.random.seed(RANDOM_STATE)

chars   = sorted(set(''.join(tr['url'].tolist())))
char2idx = {c: i + 1 for i, c in enumerate(chars)}   # 0 = pad / unknown
VOCAB   = len(char2idx) + 1
print(f'Vocab size: {VOCAB}')

def encode(urls: list[str]) -> np.ndarray:
    out = np.zeros((len(urls), MAXLEN), dtype=np.int32)
    for i, u in enumerate(urls):
        for j, c in enumerate(u[:MAXLEN]):
            out[i, j] = char2idx.get(c, 0)
    return out

X_train = encode(tr['url'].tolist());  y_train = tr['label'].values
X_test  = encode(te['url'].tolist());  y_test  = te['label'].values
print(f'Encoded  train: {X_train.shape}  test: {X_test.shape}')

Vocab size: 331
Encoded  train: (512895, 200)  test: (128224, 200)


## Step 3 — Build model
Parallel Conv1D with filter widths 3/4/5 capture different n-gram patterns in the URL character stream. Class weights compensate for the ~2:1 benign:malicious imbalance.

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)

def build_char_cnn(vocab: int, maxlen: int) -> Model:
    inp = layers.Input(shape=(maxlen,), dtype='int32')
    x   = layers.Embedding(vocab, 32, mask_zero=False)(inp)
    convs = []
    for k in (3, 4, 5):
        c = layers.Conv1D(128, k, activation='relu')(x)
        c = layers.GlobalMaxPooling1D()(c)
        convs.append(c)
    x = layers.Concatenate()(convs)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

model = build_char_cnn(VOCAB, MAXLEN)
model.summary()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('Class weights:', class_weight)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 32)   │     10,592 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 198, 128)  │     12,416 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 197, 128)  │     16,512 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 196, 128)  │     20,608 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 384)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     24,640 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 84,833 (331.38 KB)

 Trainable params: 84,833 (331.38 KB)

 Non-trainable params: 0 (0.00 B)

Class weights: {0: 0.7488305340123341, 1: 1.504699849205837}


## Step 4 — Train
5 epochs on ~513k URLs. Validation split gives early visibility on convergence.

In [5]:
t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=512,
    class_weight=class_weight,
    verbose=1,
)
train_time = time.time() - t0
print(f'\nTrained in {train_time:.1f}s ({train_time/60:.1f} min)')

Epoch 1/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 102s 113ms/step - accuracy: 0.9427 - loss: 0.1614 - val_accuracy: 0.9721 - val_loss: 0.0860
Epoch 2/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 112s 124ms/step - accuracy: 0.9691 - loss: 0.0953 - val_accuracy: 0.9781 - val_loss: 0.0685
Epoch 3/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 131s 145ms/step - accuracy: 0.9745 - loss: 0.0786 - val_accuracy: 0.9791 - val_loss: 0.0637
Epoch 4/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 145s 160ms/step - accuracy: 0.9772 - loss: 0.0697 - val_accuracy: 0.9812 - val_loss: 0.0570
Epoch 5/5
902/902 ━━━━━━━━━━━━━━━━━━━━ 180s 200ms/step - accuracy: 0.9793 - loss: 0.0639 - val_accuracy: 0.9826 - val_loss: 0.0524

Trained in 670.4s (11.2 min)


## Step 5 — Evaluate on held-out test split

In [6]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

proba = model.predict(X_test, batch_size=1024).ravel()
preds = (proba >= 0.5).astype(int)

f1 = f1_score(y_test, preds)

print('=== Test-set metrics ===')
print(f'  Accuracy  : {accuracy_score(y_test, preds):.4f}')
print(f'  Precision : {precision_score(y_test, preds):.4f}')
print(f'  Recall    : {recall_score(y_test, preds):.4f}')
print(f'  F1        : {f1:.4f}  (target ≥ 0.974)')
print(f'  ROC-AUC   : {roc_auc_score(y_test, proba):.4f}')
print()
print(classification_report(y_test, preds, target_names=['benign', 'malicious']))

assert f1 >= 0.974, f'F1 {f1:.4f} below target — investigate before uploading artifact'

126/126 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step
=== Test-set metrics ===
  Accuracy  : 0.9837
  Precision : 0.9769
  Recall    : 0.9740
  F1        : 0.9755  (target ≥ 0.974)
  ROC-AUC   : 0.9979

              precision    recall  f1-score   support

      benign       0.99      0.99      0.99     85616
   malicious       0.98      0.97      0.98     42608

    accuracy                           0.98    128224
   macro avg       0.98      0.98      0.98    128224
weighted avg       0.98      0.98      0.98    128224



## Step 6 — Save artifacts
Both files are required for inference. The `char_vocab.json` must travel with the model — inference must encode URLs with the **exact same** char→int map.

In [7]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

model_path = MODELS / 'url_charcnn.keras'
vocab_path = MODELS / 'url_char_vocab.json'

model.save(model_path)
with open(vocab_path, 'w') as f:
    json.dump({'char2idx': char2idx, 'MAXLEN': MAXLEN, 'VOCAB': VOCAB}, f)

model_digest = sha256_file(model_path)
vocab_digest = sha256_file(vocab_path)

print(f'Model : {model_path}  ({model_path.stat().st_size/1e6:.1f} MB)')
print(f'        SHA-256: {model_digest}')
print(f'Vocab : {vocab_path}')
print(f'        SHA-256: {vocab_digest}')

Model : models/url_charcnn.keras  (1.1 MB)
        SHA-256: d1df114f8f4016ac14c07378bb6d6dd3bea44f0e7490106dd1b5f92a22ac2df1
Vocab : models/url_char_vocab.json
        SHA-256: b791f91244f15c872d047c08d022f52a5b4ac849694baad623c5fd35a06abf4d


## Step 7 — Upload artifacts to S3

In [8]:
def s3_upload(local: Path, s3_path: str, profile: str):
    result = subprocess.run(
        ['aws', 's3', 'cp', str(local), s3_path, '--profile', profile],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f'  ✅ uploaded  {local.name}  →  {s3_path}')
    else:
        print(f'  ⚠️  failed   {local.name}\n{result.stderr}')
        print(f'  Run manually: aws s3 cp {local} {s3_path} --profile {profile}')

s3_upload(model_path, f'{S3_BASE}/url_charcnn.keras',     S3_PROFILE)
s3_upload(vocab_path, f'{S3_BASE}/url_char_vocab.json',   S3_PROFILE)

  ✅ uploaded  url_charcnn.keras  →  s3://email-security-pipeline-datasets/models/artifacts/url/url_charcnn.keras
  ✅ uploaded  url_char_vocab.json  →  s3://email-security-pipeline-datasets/models/artifacts/url/url_char_vocab.json


## Summary

In [9]:
print('=' * 60)
print('M6-T3 URL CHAR-CNN SUMMARY')
print('=' * 60)
print(f'Architecture : Embedding({VOCAB},32) → Conv1D(128,k=3/4/5)')
print(f'               → GlobalMaxPool → Concat → Dropout(0.4)')
print(f'               → Dense(64) → Dense(1,sigmoid)')
print(f'Params       : {model.count_params():,}')
print(f'Train rows   : {len(tr):,}  |  epochs=5  batch=512')
print(f'Train time   : {train_time:.1f}s ({train_time/60:.1f} min)')
print(f'MAXLEN       : {MAXLEN}  |  VOCAB: {VOCAB}')
print(f'Test F1      : {f1:.4f}  {"✅" if f1 >= 0.974 else "❌ below target"}')
print()
print(f'Artifacts:')
print(f'  {model_path.name:<25}  SHA-256: {model_digest}')
print(f'  {vocab_path.name:<25}  SHA-256: {vocab_digest}')
print(f'  S3: {S3_BASE}/')
print()
print('Consumed by:')
print('  M6-T6  inference wrapper (url_charcnn.keras + url_char_vocab.json)')
print('  M6-T7  artifact registry')
print('=' * 60)

M6-T3 URL CHAR-CNN SUMMARY
Architecture : Embedding(331,32) → Conv1D(128,k=3/4/5)
               → GlobalMaxPool → Concat → Dropout(0.4)
               → Dense(64) → Dense(1,sigmoid)
Params       : 84,833
Train rows   : 512,895  |  epochs=5  batch=512
Train time   : 670.4s (11.2 min)
MAXLEN       : 200  |  VOCAB: 331
Test F1      : 0.9755  ✅

Artifacts:
  url_charcnn.keras          SHA-256: d1df114f8f4016ac14c07378bb6d6dd3bea44f0e7490106dd1b5f92a22ac2df1
  url_char_vocab.json        SHA-256: b791f91244f15c872d047c08d022f52a5b4ac849694baad623c5fd35a06abf4d
  S3: s3://email-security-pipeline-datasets/models/artifacts/url/

Consumed by:
  M6-T6  inference wrapper (url_charcnn.keras + url_char_vocab.json)
  M6-T7  artifact registry
